In [1]:
import json
import random

new_data = []
data = []
data_6060 = [json.loads(i) for i in open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_eval_6060/output/gold_dev_prompt_gpt4o.jsonl", 'r').readlines()]
term_6060 = [json.loads(i) for i in open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_eval_6060/output/predictions_dev_aya_hard_replace.jsonl", 'r').readlines()]

for i, j in zip(data_6060, term_6060):
    info = i
    info['terms_dict'] = j['terms_dict']
    data.append(info)

new_data = random.sample(data, 50)

data = []

data_mmc = [json.loads(i) for i in open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/output/eval_data_gold.jsonl", 'r').readlines()]
term_mmc = [json.loads(i) for i in open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/output/aya_old_hard_replace.jsonl", 'r').readlines()]

for i, j in zip(data_mmc, term_mmc):
    info = i
    info['terms_dict'] = j['terms_dict']
    data.append(info)
    
new_data += random.sample(data, 50)

In [2]:
len(new_data), new_data[0]

(100,
 {'text': 'Please note that the beneficial impact of blockwise encoding and self attention is sustained.\n',
  'text_Chinese': '请注意，分块编码和自我注意力的有益影响是持续的。',
  'text_Arabic': 'يرجى ملاحظة أن التأثير المفيد لتشفير الكتلة والانتباه الذاتي مستمر.',
  'text_French': 'Veuillez noter que l’impact bénéfique de l’encodage par blocs et de l’Attention personnelle est maintenu.',
  'text_Japanese': 'ブロックワイズのエンコーディングとセルフ注意の有益な影響は維持されることに注意してください。',
  'text_Russian': 'Обратите внимание, что благотворное влияние кодировки посредством Blockwise и самовнимания довольно стабильно.',
  'terms_dict': {'attention': {'Chinese': '注意力',
    'Arabic': 'الانتباه',
    'French': 'Attention',
    'Japanese': '注意',
    'Russian': 'Внимание'},
   'encoding': {'Chinese': '编码',
    'Arabic': 'تشفير',
    'French': 'encodage',
    'Japanese': 'エンコーディング',
    'Russian': 'кодировка'}}})

create mturk data:
- a dataframe including "en_sent". "terms", "trans_sent"

In [7]:
import pandas as pd

tgt_langs = [
    "Arabic",
    "Chinese",
    "French",
    "Japanese",
    "Russian"
]

for tgt_lang in tgt_langs:
    infos = []
    for item in new_data:
        terms = {}
        for en_term, tgt_term in item['terms_dict'].items():
            if tgt_lang in tgt_term:
                terms[en_term] = tgt_term[tgt_lang]
            
        terms_str = "; ".join([f"{key}: {val}" for key, val in terms.items()])
        
        infos.append({
            "en_sent": item['text'],
            "trans_sent": item[f'text_{tgt_lang}'],
            "terms": terms
        })
    df = pd.DataFrame(infos)
    df.to_csv(f"{tgt_lang}.csv", index=False)